# Exploration des données - Détection de Fraudes Financières
**Datasets** : PaySim (transactions mobiles) + IBM AML HI-Small (blanchiment réel)

In [ ]:
import sys, os
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
from config.config import PAYSIM_PATH, IBM_AML_TRANS, IBM_AML_ACCOUNTS

## 1. Dataset PaySim (streaming Kafka)

In [ ]:
paysim = pd.read_csv(PAYSIM_PATH)
print(f'Dimensions : {paysim.shape}')
paysim.head(3)

In [ ]:
# Distribution des types de transactions
fig, axes = plt.subplots(1, 2)
paysim['type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', title='Types de transactions')
fraud_counts = paysim['isFraud'].value_counts()
axes[1].pie(fraud_counts, labels=['Normal', 'Fraude'], autopct='%1.2f%%', colors=['#2196F3','#F44336'])
axes[1].set_title('Répartition fraudes / normales')
plt.tight_layout()
plt.show()
print(f"Total transactions : {len(paysim):,} | Fraudes : {paysim['isFraud'].sum():,} ({paysim['isFraud'].mean()*100:.2f}%)")

In [ ]:
# Distribution des montants (fraudes vs normales)
import numpy as np
fig, axes = plt.subplots(1, 2)
paysim[paysim['isFraud']==0]['amount'].clip(upper=1e6).hist(bins=50, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Montants - Transactions normales'); axes[0].set_xlabel('Montant')
paysim[paysim['isFraud']==1]['amount'].clip(upper=1e6).hist(bins=50, ax=axes[1], color='red', alpha=0.7)
axes[1].set_title('Montants - Transactions frauduleuses'); axes[1].set_xlabel('Montant')
plt.tight_layout(); plt.show()

## 2. Dataset IBM AML HI-Small (graphe de blanchiment)

In [ ]:
aml = pd.read_csv(IBM_AML_TRANS)
print(f'Dimensions : {aml.shape}')
print(f'Colonnes : {list(aml.columns)}')
aml.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

launder_counts = aml['Is Laundering'].value_counts()
axes[0].pie(launder_counts, labels=['Normal','Blanchiment'], autopct='%1.2f%%', colors=['#4CAF50','#FF9800'])
axes[0].set_title('Blanchiment vs Normal')

aml['Payment Format'].value_counts().plot(kind='bar', ax=axes[1], color='teal', title='Formats de paiement')
axes[1].tick_params(axis='x', rotation=45)

aml['Payment Currency'].value_counts().head(10).plot(kind='bar', ax=axes[2], color='coral', title='Top 10 devises')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()
print(f"Transactions de blanchiment : {launder_counts.get(1,0):,} ({launder_counts.get(1,0)/len(aml)*100:.2f}%)")

In [ ]:
# Statistiques globales
print('=== RÉSUMÉ DES DONNÉES ===')
print(f'PaySim  : {len(paysim):>10,} transactions | {paysim["isFraud"].sum():>6,} fraudes ({paysim["isFraud"].mean()*100:.2f}%)')
print(f'IBM AML : {len(aml):>10,} transactions | {aml["Is Laundering"].sum():>6,} blanchiments ({aml["Is Laundering"].mean()*100:.2f}%)')
nb_comptes = pd.concat([aml['Account'], aml['Account.1']]).nunique()
print(f'Comptes uniques IBM AML : {nb_comptes:,}')